# Delta Demo — Day 1: Initial Load
Creates the schema + Volume (safe to re-run, uses `IF NOT EXISTS`), writes the first 3 rows as a brand-new Delta table, then verifies everything: the raw files on disk, a normal query, `DESCRIBE HISTORY`, and a full raw-contents deep dive of the JSON commit and Parquet file. If you want a clean slate before this, run **Delta_Demo_Day0_Load** first.

## Step 0 — Setup
We need two things before any Delta table can exist: a **schema** (a container/folder for organizing objects, like a database) and a **Volume** (Unity Catalog's managed, browsable file storage area). We're using a Volume — not a regular managed table — specifically so we can run `%sh ls` / `%sh cat` later and actually SEE the raw files Delta writes. A normal managed table hides its physical storage path from users.

In [0]:
%sql
-- Creates the schema (if it doesn't already exist) that will hold our Volume.
CREATE SCHEMA IF NOT EXISTS workspace.delta_demo;

-- Creates a Volume named 'demo_files' inside that schema.
-- A Volume is just a folder Databricks manages for you — anything we put inside it
-- is real, ordinary file storage we can browse with normal shell commands.
CREATE VOLUME IF NOT EXISTS workspace.delta_demo.demo_files;

## Day 1 — Create Path-Based Delta Table (Initial Load)
This is the very first write. We are NOT running `CREATE TABLE` in SQL — instead we build a Spark DataFrame in Python and `.save()` it directly to a folder path inside our Volume. The moment this write completes, Delta Lake creates a `_delta_log/` folder next to the data — **that folder is what makes this a "Delta table" instead of just some Parquet files sitting in storage.**

In [0]:
%python
from pyspark.sql import functions as F

# STEP 1: Build an in-memory Spark DataFrame with our starting data.
# At this point nothing has been written to disk yet — this is pure Python/Spark,
# no Delta table exists.
# Note: 'sal' is created as a plain INT first, then explicitly cast to DECIMAL(10,2)
# below — Spark Connect will NOT silently convert an int into a Decimal for you,
# so this cast step is required, not optional.
day1 = spark.createDataFrame(
    [
        (1, 'Ravi', 25000), 
        (2, 'Sridevi', 23000), 
        (3, 'Uma', 35000)],
    "eno INT, ename STRING, sal INT"
).withColumn("sal", F.col("sal").cast("DECIMAL(10,2)"))

# STEP 2: Write this DataFrame out as a Delta table at a specific path.
# mode('overwrite') here just means "create fresh" since nothing exists at this path yet.
# THIS is the line that actually creates the Delta table and its _delta_log folder —
# everything before this line was just an in-memory DataFrame.
day1.write.format("delta").mode("overwrite") \
    .save("/Volumes/workspace/delta_demo/demo_files/employees")

## Verify Table Location & Raw Files
Let's prove the table really exists by looking at the raw folder structure directly with shell commands — no Spark, no SQL, just the file system. If you're new to Delta: notice there's a `_delta_log/` folder AND a `.parquet` file sitting side by side. The Parquet file holds the actual row data; `_delta_log/` holds the transaction history that makes this a *Delta* table instead of a plain Parquet file.

In [0]:
%sh
# List everything inside _delta_log/ — this is the transaction log folder.
# Right now (version 0) you should see exactly one .json file and its .crc checksum.
ls -la /Volumes/workspace/delta_demo/demo_files/employees/_delta_log/

In [0]:
%sh
# List the table's root folder — this shows the actual Parquet data file(s)
# sitting alongside the _delta_log folder.
ls -la /Volumes/workspace/delta_demo/demo_files/employees/

## Query the Table (Day 1)
Now let's read the table the normal way — through SQL, using the `delta.`path`` syntax (this is how you query a Delta table that isn't registered in a catalog, by pointing straight at its folder path).

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees` ORDER BY eno;

## Describe History — After Day 1
`DESCRIBE HISTORY` is a Delta-specific command that shows every transaction ("commit") ever made to this table — who did what, when, and how many rows/files were affected. Right now you should see exactly ONE row: our initial WRITE.

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees`;

## Day 1 — Raw Contents Deep Dive (What's Really Inside the JSON & Parquet Files)
This is the core "look under the hood" section. We're going to read the ACTUAL bytes Delta wrote to disk — not through Delta's own reader, but directly, so you can see exactly what a transaction log commit and a data file really contain.

**Important gotcha for beginners:** each `_delta_log/*.json` file is NOT one big JSON object — it's **newline-delimited JSON (NDJSON)**, meaning every line is its own separate, complete JSON object (one per "action": commitInfo, metaData, protocol, add, etc). If you pipe the whole file into a normal JSON pretty-printer at once, it will error out after the first line — that's expected, not a bug.

In [0]:
%sh
# This command reads version 0's commit file line by line, and pretty-prints EACH
# line separately (since each line is its own independent JSON object).
# You should see 4 separate blocks: commitInfo, metaData, protocol, and add.
cat /Volumes/workspace/delta_demo/demo_files/employees/_delta_log/00000000000000000000.json \
  | while IFS= read -r line; do echo "$line" | python3 -m json.tool; echo "---"; done

In [0]:
%python
# Now let's read the ACTUAL Parquet file directly — bypassing Delta entirely.
# We deliberately use pandas here instead of spark.read.parquet(...), because Spark
# will actually REFUSE to read a raw Parquet file that lives inside a Delta table's
# folder (it forces you to go through format('delta') instead, to stop you from
# accidentally reading stale/incomplete data). Pandas has no such protection, so it
# lets us see the raw physical bytes exactly as they are on disk.
import pandas as pd
import glob

parquet_files = glob.glob("/Volumes/workspace/delta_demo/demo_files/employees/*.parquet")
for f in sorted(parquet_files):
    print(f"\n=== {f.split('/')[-1]} ===")
    print(pd.read_parquet(f))

## Day 1 Summary — List All Objects
A full listing of every object that exists on disk at the end of Day 1: the transaction log folder and the data folder, side by side.

In [0]:
%sh
echo "=== _delta_log/ ==="
ls -la /Volumes/workspace/delta_demo/demo_files/employees/_delta_log/
echo ""
echo "=== data files (Parquet + deletion vectors) ==="
ls -la /Volumes/workspace/delta_demo/demo_files/employees/ | grep -v _delta_log